In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from IPython.display import display

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

from pathlib import Path

def find_data_dir() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        candidate = base / "Data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find Data/ directory. Run the notebook from the repo root, Models/, or DataExploration/."
    )

DATA_DIR = find_data_dir()

try:
    file_path = DATA_DIR / "humsavar_dbnsfp53_merged_fast.csv"
    df = pd.read_csv(file_path, na_values="?")
    print(f"Successfully loaded data from {file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the file exists in the repo Data/ directory.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
def clean_score(value):
    """
    Converts semicolon-separated score strings into a single float.
    Example: '0.698;0.698;.;0.698' -> 0.698
    """
    if pd.isna(value):
        return np.nan

    parts = str(value).split(";")

    for p in parts:
        p = p.strip()
        if p != "." and p != "":
            try:
                return float(p)
            except:
                continue

    return np.nan


score_cols = {
    "SIFT_score": "SIFT_score_clean",
    "Polyphen2_HDIV_score": "PolyPhen_score_clean",
    "CADD_raw": "CADD_raw_clean",
    "CADD_phred": "CADD_phred_clean",
    "REVEL_score": "REVEL_score_clean"
}

for old_col, new_col in score_cols.items():
    df[new_col] = df[old_col].apply(clean_score)

df[[
    "SIFT_score_clean",
    "PolyPhen_score_clean",
    "CADD_raw_clean",
    "CADD_phred_clean",
    "REVEL_score_clean"
]].head()

In [ ]:

df["y"] = df["Label"].map({
    "Benign": 0,
    "Pathogenic": 1
})

df = df.dropna(subset=["y"])

df["y"].value_counts()


In [ ]:
features = [
    "SIFT_score_clean",
    "PolyPhen_score_clean",
    "CADD_raw_clean",
    "CADD_phred_clean",
    "REVEL_score_clean"
]

model_df = df.dropna(subset=features + ["y"]).copy()

X = model_df[features]
y = model_df["y"]

print("Rows used:", len(model_df))
print(y.value_counts())

model_path = DATA_DIR / "model_df.csv"
model_df.to_csv(model_path, index=False)
print(f"Saved {model_path}")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X,
    y,
    model_df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

predictor_scores = {
    "SIFT": 1 - model_df.loc[test_idx, "SIFT_score_clean"],
    "PolyPhen-2": model_df.loc[test_idx, "PolyPhen_score_clean"],
    "CADD raw": model_df.loc[test_idx, "CADD_raw_clean"],
    "CADD phred": model_df.loc[test_idx, "CADD_phred_clean"],
    "REVEL": model_df.loc[test_idx, "REVEL_score_clean"]
}

results = []

for name, scores in predictor_scores.items():
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)

    results.append({
        "Model/Predictor": name,
        "ROC-AUC": auc,
        "Average Precision": ap
    })

benchmark_results = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
benchmark_results

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

threshold_predictions = {
    "SIFT threshold": (model_df.loc[test_idx, "SIFT_score_clean"] < 0.05).astype(int),
    "PolyPhen-2 threshold": (model_df.loc[test_idx, "PolyPhen_score_clean"] > 0.909).astype(int),
    "CADD PHRED threshold": (model_df.loc[test_idx, "CADD_phred_clean"] > 20).astype(int),
    "REVEL threshold": (model_df.loc[test_idx, "REVEL_score_clean"] > 0.5).astype(int)
}

threshold_results = []

for name, preds in threshold_predictions.items():
    threshold_results.append({
        "Predictor": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds)
    })

threshold_results = pd.DataFrame(threshold_results).sort_values("F1", ascending=False)
threshold_results

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

log_reg.fit(X_train, y_train)

log_reg_probs = log_reg.predict_proba(X_test)[:, 1]
log_reg_preds = log_reg.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, log_reg_probs))
print("Average Precision:", average_precision_score(y_test, log_reg_probs))
print(classification_report(y_test, log_reg_preds))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]
rf_preds = rf.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, rf_probs))
print("Average Precision:", average_precision_score(y_test, rf_probs))
print(classification_report(y_test, rf_preds))

In [ ]:
ml_results = []

models = {
    "Logistic Regression": log_reg_probs,
    "Random Forest": rf_probs
}

for name, probs in models.items():
    preds = (probs >= 0.5).astype(int)

    ml_results.append({
        "Model/Predictor": name,
        "ROC-AUC": roc_auc_score(y_test, probs),
        "Average Precision": average_precision_score(y_test, probs),
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds)
    })

for name, scores in predictor_scores.items():
    preds = (scores >= 0.5).astype(int)

    ml_results.append({
        "Model/Predictor": name,
        "ROC-AUC": roc_auc_score(y_test, scores),
        "Average Precision": average_precision_score(y_test, scores),
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "F1": np.nan
    })

all_results = pd.DataFrame(ml_results).sort_values("ROC-AUC", ascending=False)
all_results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(all_results["Model/Predictor"], all_results["ROC-AUC"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("ROC-AUC")
plt.title("Benchmark Comparison of Pathogenicity Predictors and ML Models")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))

# Existing predictors
for name, scores in predictor_scores.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc:.3f}")

# ML models
for name, probs in models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
failure_df = model_df.loc[test_idx].copy()

failure_df["SIFT_score_for_auc"] = 1 - failure_df["SIFT_score_clean"]
failure_df["PolyPhen_score_for_auc"] = failure_df["PolyPhen_score_clean"]
failure_df["CADD_phred_score_for_auc"] = failure_df["CADD_phred_clean"]
failure_df["REVEL_score_for_auc"] = failure_df["REVEL_score_clean"]

failure_df["RF_prob_pathogenic"] = rf_probs
failure_df["LogReg_prob_pathogenic"] = log_reg_probs

failure_df.head()

In [ ]:
failure_df["SIFT_pred_binary"] = (failure_df["SIFT_score_clean"] < 0.05).astype(int)
failure_df["PolyPhen_pred_binary"] = (failure_df["PolyPhen_score_clean"] > 0.909).astype(int)
failure_df["CADD_pred_binary"] = (failure_df["CADD_phred_clean"] > 20).astype(int)
failure_df["REVEL_pred_binary"] = (failure_df["REVEL_score_clean"] > 0.5).astype(int)
failure_df["RF_pred_binary"] = (failure_df["RF_prob_pathogenic"] >= 0.5).astype(int)

failure_df["SIFT_failed"] = failure_df["SIFT_pred_binary"] != failure_df["y"]
failure_df["PolyPhen_failed"] = failure_df["PolyPhen_pred_binary"] != failure_df["y"]
failure_df["CADD_failed"] = failure_df["CADD_pred_binary"] != failure_df["y"]
failure_df["REVEL_failed"] = failure_df["REVEL_pred_binary"] != failure_df["y"]
failure_df["RF_failed"] = failure_df["RF_pred_binary"] != failure_df["y"]

failure_df[[
    "Gene",
    "AA_change",
    "Label",
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed",
    "RF_failed"
]].head()

In [ ]:
failure_cols = [
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed"
]

failure_df["num_predictors_failed"] = failure_df[failure_cols].sum(axis=1)

hard_cases = failure_df.sort_values("num_predictors_failed", ascending=False)

hard_cases[[
    "Gene",
    "AA_change",
    "Label",
    "Disease",
    "SIFT_score_clean",
    "PolyPhen_score_clean",
    "CADD_phred_clean",
    "REVEL_score_clean",
    "num_predictors_failed"
]].head(20)

In [ ]:
gene_failure = (
    failure_df
    .groupby("Gene")
    .agg(
        total_variants=("Gene", "count"),
        avg_predictors_failed=("num_predictors_failed", "mean"),
        variants_with_any_failure=("num_predictors_failed", lambda x: (x > 0).sum())
    )
    .reset_index()
)

gene_failure["any_failure_rate"] = (
    gene_failure["variants_with_any_failure"] / gene_failure["total_variants"]
)

gene_failure = gene_failure[gene_failure["total_variants"] >= 5]

gene_failure.sort_values(
    ["any_failure_rate", "avg_predictors_failed", "total_variants"],
    ascending=False
).head(20)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Create binary predictions for all benchmark methods
benchmark_binary_preds = {
    "SIFT": (failure_df["SIFT_score_clean"] < 0.05).astype(int),
    "PolyPhen-2": (failure_df["PolyPhen_score_clean"] > 0.909).astype(int),
    "CADD PHRED": (failure_df["CADD_phred_clean"] > 20).astype(int),
    "REVEL": (failure_df["REVEL_score_clean"] > 0.5).astype(int),
    "Logistic Regression": (failure_df["LogReg_prob_pathogenic"] >= 0.5).astype(int),
    "Random Forest": (failure_df["RF_prob_pathogenic"] >= 0.5).astype(int)
}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (name, preds) in zip(axes, benchmark_binary_preds.items()):
    cm = confusion_matrix(y_test, preds)

    im = ax.imshow(cm, cmap="Blues")

    # Add numbers inside each box
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm[i, j]

            # Make text readable depending on background color
            text_color = "white" if value > cm.max() / 2 else "black"

            ax.text(
                j,
                i,
                str(value),
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold",
                color=text_color
            )

    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Benign", "Pathogenic"], fontsize=10)
    ax.set_yticklabels(["Benign", "Pathogenic"], fontsize=10)

    # Remove unnecessary borders
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle(
    "Confusion Matrices Across Benchmark Predictors",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison_rows = []

# Probability/score-based models for ROC-AUC and PR-AUC
score_based_outputs = {
    "SIFT": 1 - failure_df["SIFT_score_clean"],
    "PolyPhen-2": failure_df["PolyPhen_score_clean"],
    "CADD PHRED": failure_df["CADD_phred_clean"],
    "REVEL": failure_df["REVEL_score_clean"],
    "Logistic Regression": failure_df["LogReg_prob_pathogenic"],
    "Random Forest": failure_df["RF_prob_pathogenic"]
}

for name in benchmark_binary_preds.keys():
    preds = benchmark_binary_preds[name]
    scores = score_based_outputs[name]

    comparison_rows.append({
        "Model/Predictor": name,
        "ROC-AUC": roc_auc_score(y_test, scores),
        "PR-AUC": average_precision_score(y_test, scores),
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "False Positives": confusion_matrix(y_test, preds)[0, 1],
        "False Negatives": confusion_matrix(y_test, preds)[1, 0]
    })

benchmark_table = pd.DataFrame(comparison_rows)
benchmark_table = benchmark_table.sort_values("ROC-AUC", ascending=False)

benchmark_table

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    benchmark_table["Model/Predictor"],
    benchmark_table["ROC-AUC"]
)

plt.ylabel("ROC-AUC")
plt.xlabel("Model / Predictor")
plt.title("ROC-AUC Comparison Across Benchmarks")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    benchmark_table["Model/Predictor"],
    benchmark_table["F1"]
)

plt.ylabel("F1 Score")
plt.xlabel("Model / Predictor")
plt.title("F1 Score Comparison Across Benchmarks")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    benchmark_table["Recall"],
    benchmark_table["Precision"]
)

for _, row in benchmark_table.iterrows():
    plt.text(
        row["Recall"] + 0.005,
        row["Precision"] + 0.005,
        row["Model/Predictor"],
        fontsize=9
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision vs Recall by Benchmark")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
error_counts = benchmark_table[[
    "Model/Predictor",
    "False Positives",
    "False Negatives"
]].copy()

error_counts.plot(
    x="Model/Predictor",
    kind="bar",
    figsize=(10, 5)
)

plt.ylabel("Number of Errors")
plt.title("False Positives vs False Negatives by Benchmark")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(8, 6))

for name, scores in score_based_outputs.items():
    precision, recall, _ = precision_recall_curve(y_test, scores)
    pr_auc = average_precision_score(y_test, scores)

    plt.plot(recall, precision, label=f"{name} PR-AUC={pr_auc:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves Across Benchmarks")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
score_columns_for_plot = {
    "SIFT": "SIFT_score_clean",
    "PolyPhen-2": "PolyPhen_score_clean",
    "CADD PHRED": "CADD_phred_clean",
    "REVEL": "REVEL_score_clean",
    "Random Forest": "RF_prob_pathogenic",
    "Logistic Regression": "LogReg_prob_pathogenic"
}

for name, col in score_columns_for_plot.items():
    plt.figure(figsize=(8, 5))

    plt.hist(
        failure_df[failure_df["y"] == 0][col],
        bins=40,
        alpha=0.6,
        label="Benign"
    )

    plt.hist(
        failure_df[failure_df["y"] == 1][col],
        bins=40,
        alpha=0.6,
        label="Pathogenic"
    )

    plt.xlabel("Score")
    plt.ylabel("Count")
    plt.title(f"Score Distribution by True Label: {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
corr_cols = [
    "SIFT_score_clean",
    "PolyPhen_score_clean",
    "CADD_phred_clean",
    "REVEL_score_clean",
    "RF_prob_pathogenic",
    "LogReg_prob_pathogenic"
]

corr_matrix = failure_df[corr_cols].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix)
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(corr_cols)),
    corr_cols,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(corr_cols)),
    corr_cols
)

plt.title("Correlation Between Predictor Scores")
plt.tight_layout()
plt.show()

In [ ]:
failure_df["SIFT_flipped"] = 1 - failure_df["SIFT_score_clean"]

corr_cols_flipped = [
    "SIFT_flipped",
    "PolyPhen_score_clean",
    "CADD_phred_clean",
    "REVEL_score_clean",
    "RF_prob_pathogenic",
    "LogReg_prob_pathogenic"
]

corr_matrix_flipped = failure_df[corr_cols_flipped].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix_flipped)
plt.colorbar(label="Correlation")

for i in range(len(corr_cols_flipped)):
    for j in range(len(corr_cols_flipped)):
        plt.text(
            j,
            i,
            f"{corr_matrix_flipped.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.xticks(
    range(len(corr_cols_flipped)),
    ["SIFT flipped", "PolyPhen", "CADD", "REVEL", "RF", "LogReg"],
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(corr_cols_flipped)),
    ["SIFT flipped", "PolyPhen", "CADD", "REVEL", "RF", "LogReg"]
)

plt.title("Predictor Score Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    rf_importance["Feature"],
    rf_importance["Importance"]
)

plt.ylabel("Feature Importance")
plt.xlabel("Feature")
plt.title("Random Forest Feature Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

failure_overlap_cols = [
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed",
    "RF_failed"
]

labels = ["SIFT", "PolyPhen", "CADD", "REVEL", "RF"]

failure_overlap = failure_df[failure_overlap_cols].astype(int).corr()
failure_overlap.index = labels
failure_overlap.columns = labels

plt.figure(figsize=(7, 6))

ax = sns.heatmap(
    failure_overlap,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Failure Correlation", "shrink": 0.9},
    annot_kws={"size": 12, "weight": "bold"}
)

plt.title("Failure Overlap Between Predictors", fontsize=15, pad=12)
plt.xticks(rotation=0, fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

failure_df["num_predictors_failed"].value_counts().sort_index().plot(kind="bar")

plt.xlabel("Number of Predictors Failed")
plt.ylabel("Number of Variants")
plt.title("How Many Predictors Fail Per Variant?")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
top_gene_failure = gene_failure.sort_values(
    "any_failure_rate",
    ascending=False
).head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_gene_failure["Gene"],
    top_gene_failure["any_failure_rate"]
)

plt.xlabel("Failure Rate")
plt.ylabel("Gene")
plt.title("Top Genes by Predictor Failure Rate")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
top_genes = (
    gene_failure
    .sort_values(
        ["any_failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(10)
)
top_genes_path = DATA_DIR / "top_10_failure_genes.csv"
top_genes.to_csv(top_genes_path, index=False)
print(f"Saved {top_genes_path}")
top_genes


In [ ]:
top_gene_names = top_genes["Gene"].tolist()

alphafold_variants = failure_df[
    failure_df["Gene"].isin(top_gene_names)
].copy()

alphafold_variants[[
    "Gene",
    "Entry",
    "AA_change",
    "Label",
    "Disease",
    "num_predictors_failed",
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed"
]].head(20)

alphafold_path = DATA_DIR / "alphafold_variants.csv"
alphafold_variants.to_csv(alphafold_path, index=False)
print(f"Saved {alphafold_path}")


In [ ]:
# Core failure columns from your notebook
failure_cols = [
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed"
]

# Make sure failures are boolean
for col in failure_cols:
    failure_df[col] = failure_df[col].astype(bool)

# Any failure among the four traditional predictors
failure_df["any_failure"] = failure_df[failure_cols].any(axis=1)

# Number of predictors that failed
failure_df["num_predictors_failed"] = failure_df[failure_cols].sum(axis=1)

failure_df[[
    "Gene",
    "AA_change",
    "Label",
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed",
    "any_failure",
    "num_predictors_failed"
]].head()

In [ ]:
# Core failure columns from your notebook
failure_cols = [
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed"
]

# Make sure failures are boolean
for col in failure_cols:
    failure_df[col] = failure_df[col].astype(bool)

# Any failure among the four traditional predictors
failure_df["any_failure"] = failure_df[failure_cols].any(axis=1)

# Number of predictors that failed
failure_df["num_predictors_failed"] = failure_df[failure_cols].sum(axis=1)

failure_df[[
    "Gene",
    "AA_change",
    "Label",
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed",
    "any_failure",
    "num_predictors_failed"
]].head()

In [ ]:
gene_failure = (
    failure_df
    .groupby("Gene")
    .agg(
        total_variants=("Gene", "count"),
        variants_with_any_failure=("any_failure", "sum"),
        any_failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean"),
        SIFT_failure_rate=("SIFT_failed", "mean"),
        PolyPhen_failure_rate=("PolyPhen_failed", "mean"),
        CADD_failure_rate=("CADD_failed", "mean"),
        REVEL_failure_rate=("REVEL_failed", "mean")
    )
    .reset_index()
)

# Avoid genes with tiny sample sizes
gene_failure_filtered = gene_failure[gene_failure["total_variants"] >= 5].copy()

top_genes = (
    gene_failure_filtered
    .sort_values(
        ["any_failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(10)
)

top_genes

In [ ]:
gene_heatmap = top_genes.set_index("Gene")[[
    "SIFT_failure_rate",
    "PolyPhen_failure_rate",
    "CADD_failure_rate",
    "REVEL_failure_rate"
]]

plt.figure(figsize=(9, 6))

sns.heatmap(
    gene_heatmap,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    vmin=0,
    vmax=1
)

plt.title("Predictor Failure Rates Across High-Failure Genes")
plt.xlabel("Predictor")
plt.ylabel("Gene")
plt.tight_layout()
plt.show()

In [ ]:
import re

def extract_aa_substitution(aa_change):
    """
    Extracts amino acid substitution from strings like:
    p.Arg152His
    p.Gly162Ser
    """
    if pd.isna(aa_change):
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    match = re.match(r"p\.([A-Za-z]{3})(\d+)([A-Za-z]{3})", str(aa_change))

    if match:
        ref_aa = match.group(1)
        position = int(match.group(2))
        alt_aa = match.group(3)
        substitution = f"{ref_aa}->{alt_aa}"
        return pd.Series([ref_aa, position, alt_aa, substitution])

    return pd.Series([np.nan, np.nan, np.nan, np.nan])

failure_df[["ref_aa", "aa_position", "alt_aa", "aa_substitution"]] = (
    failure_df["AA_change"].apply(extract_aa_substitution)
)

failure_df[[
    "AA_change",
    "ref_aa",
    "aa_position",
    "alt_aa",
    "aa_substitution"
]].head()

In [ ]:
failed_variants = failure_df[failure_df["any_failure"] == True].copy()

top_failed_substitutions = (
    failed_variants["aa_substitution"]
    .value_counts()
    .reset_index()
)

top_failed_substitutions.columns = ["aa_substitution", "failure_count"]

top_failed_substitutions.head(15)

In [ ]:
failed_variants = failure_df[failure_df["any_failure"] == True].copy()

top_failed_substitutions = (
    failed_variants["aa_substitution"]
    .value_counts()
    .reset_index()
)

top_failed_substitutions.columns = ["aa_substitution", "failure_count"]

top_failed_substitutions.head(15)

In [ ]:
substitution_failure = (
    failure_df
    .dropna(subset=["aa_substitution"])
    .groupby("aa_substitution")
    .agg(
        total_variants=("aa_substitution", "count"),
        variants_with_any_failure=("any_failure", "sum"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean")
    )
    .reset_index()
)

# Filter to substitutions with enough examples
substitution_failure_filtered = substitution_failure[
    substitution_failure["total_variants"] >= 5
].copy()

top_substitution_failures = (
    substitution_failure_filtered
    .sort_values(
        ["failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(15)
)

top_substitution_failures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = top_substitution_failures.sort_values("failure_rate", ascending=True)

plt.figure(figsize=(10, 7))

ax = sns.barplot(
    data=plot_df,
    x="failure_rate",
    y="aa_substitution"
)

plt.title("Top Amino Acid Substitutions by Predictor Failure Rate")
plt.xlabel("Failure Rate")
plt.ylabel("Amino Acid Substitution")

plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x:.0%}")
)

# Add total variant count labels
for i, row in enumerate(plot_df.itertuples()):
    ax.text(
        row.failure_rate + 0.01,
        i,
        f"n={row.total_variants}",
        va="center"
    )

plt.xlim(0, min(1.1, plot_df["failure_rate"].max() + 0.15))
plt.tight_layout()
plt.show()

In [ ]:
aa_properties = {
    "Gly": "small/nonpolar",
    "Ala": "small/nonpolar",
    "Val": "hydrophobic",
    "Leu": "hydrophobic",
    "Ile": "hydrophobic",
    "Met": "hydrophobic",
    "Phe": "aromatic",
    "Trp": "aromatic",
    "Tyr": "aromatic/polar",
    "Ser": "polar",
    "Thr": "polar",
    "Cys": "polar/sulfur",
    "Asn": "polar",
    "Gln": "polar",
    "Lys": "positive",
    "Arg": "positive",
    "His": "positive/aromatic",
    "Asp": "negative",
    "Glu": "negative",
    "Pro": "special/rigid"
}

failure_df["ref_aa_property"] = failure_df["ref_aa"].map(aa_properties)
failure_df["alt_aa_property"] = failure_df["alt_aa"].map(aa_properties)

failure_df["biochemical_change"] = (
    failure_df["ref_aa_property"] + " -> " + failure_df["alt_aa_property"]
)

biochemical_failure = (
    failure_df
    .dropna(subset=["biochemical_change"])
    .groupby("biochemical_change")
    .agg(
        total_variants=("biochemical_change", "count"),
        variants_with_any_failure=("any_failure", "sum"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean")
    )
    .reset_index()
)

biochemical_failure_filtered = biochemical_failure[
    biochemical_failure["total_variants"] >= 10
].copy()

top_biochemical_failures = (
    biochemical_failure_filtered
    .sort_values(
        ["failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(15)
)

top_biochemical_failures

In [ ]:
plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_biochemical_failures,
    x="failure_rate",
    y="biochemical_change"
)

plt.title("Biochemical Changes with Highest Predictor Failure Rates")
plt.xlabel("Failure Rate")
plt.ylabel("Biochemical Change")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
disease_df = failure_df.copy()

# Remove missing or empty disease annotations
disease_df = disease_df[
    disease_df["Disease"].notna() &
    (disease_df["Disease"] != "-")
].copy()

disease_failure = (
    disease_df
    .groupby("Disease")
    .agg(
        total_variants=("Disease", "count"),
        variants_with_any_failure=("any_failure", "sum"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean"),
        SIFT_failure_rate=("SIFT_failed", "mean"),
        PolyPhen_failure_rate=("PolyPhen_failed", "mean"),
        CADD_failure_rate=("CADD_failed", "mean"),
        REVEL_failure_rate=("REVEL_failed", "mean")
    )
    .reset_index()
)

# Filter to diseases with enough examples
disease_failure_filtered = disease_failure[
    disease_failure["total_variants"] >= 5
].copy()

top_disease_failures = (
    disease_failure_filtered
    .sort_values(
        ["failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(10)
)

top_disease_failures

In [ ]:
disease_df = failure_df.copy()

# Remove missing or empty disease annotations
disease_df = disease_df[
    disease_df["Disease"].notna() &
    (disease_df["Disease"] != "-")
].copy()

disease_failure = (
    disease_df
    .groupby("Disease")
    .agg(
        total_variants=("Disease", "count"),
        variants_with_any_failure=("any_failure", "sum"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean"),
        SIFT_failure_rate=("SIFT_failed", "mean"),
        PolyPhen_failure_rate=("PolyPhen_failed", "mean"),
        CADD_failure_rate=("CADD_failed", "mean"),
        REVEL_failure_rate=("REVEL_failed", "mean")
    )
    .reset_index()
)

# Filter to diseases with enough examples
disease_failure_filtered = disease_failure[
    disease_failure["total_variants"] >= 5
].copy()

top_disease_failures = (
    disease_failure_filtered
    .sort_values(
        ["failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(10)
)

top_disease_failures

In [ ]:
plt.figure(figsize=(20, 7))

sns.barplot(
    data=top_disease_failures,
    x="failure_rate",
    y="Disease"
)

plt.title("Disease Annotations with Highest Predictor Failure Rates")
plt.xlabel("Any Predictor Failure Rate")
plt.ylabel("Disease")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
prediction_cols = [
    "SIFT_pred_binary",
    "PolyPhen_pred_binary",
    "CADD_pred_binary",
    "REVEL_pred_binary"
]

# Number of predictors voting pathogenic
failure_df["num_pathogenic_votes"] = failure_df[prediction_cols].sum(axis=1)

# Disagreement means the four predictors do not all agree
# 0 votes = all benign
# 4 votes = all pathogenic
# 1, 2, 3 votes = disagreement
failure_df["predictor_disagreement"] = failure_df["num_pathogenic_votes"].between(1, 3)

failure_df[[
    "Gene",
    "AA_change",
    "Label",
    "num_pathogenic_votes",
    "predictor_disagreement",
    "any_failure",
    "num_predictors_failed"
]].head()

In [ ]:
disagreement_summary = (
    failure_df
    .groupby("predictor_disagreement")
    .agg(
        total_variants=("predictor_disagreement", "count"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean")
    )
    .reset_index()
)

disagreement_summary

In [ ]:
disagreement_summary = (
    failure_df
    .groupby("predictor_disagreement")
    .agg(
        total_variants=("predictor_disagreement", "count"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean")
    )
    .reset_index()
)

disagreement_summary

In [ ]:
vote_pattern_summary = (
    failure_df
    .groupby("num_pathogenic_votes")
    .agg(
        total_variants=("num_pathogenic_votes", "count"),
        failure_rate=("any_failure", "mean"),
        avg_predictors_failed=("num_predictors_failed", "mean")
    )
    .reset_index()
)

vote_pattern_summary

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(
    data=vote_pattern_summary,
    x="num_pathogenic_votes",
    y="failure_rate"
)

plt.title("Failure Rate by Number of Pathogenic Predictor Votes")
plt.xlabel("Number of Predictors Voting Pathogenic")
plt.ylabel("Failure Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Distance from threshold
# Smaller distance means closer to the decision boundary

failure_df["SIFT_boundary_distance"] = abs(failure_df["SIFT_score_clean"] - 0.05)
failure_df["PolyPhen_boundary_distance"] = abs(failure_df["PolyPhen_score_clean"] - 0.909)
failure_df["CADD_boundary_distance"] = abs(failure_df["CADD_phred_clean"] - 20)
failure_df["REVEL_boundary_distance"] = abs(failure_df["REVEL_score_clean"] - 0.5)

failure_df[[
    "SIFT_score_clean",
    "SIFT_boundary_distance",
    "PolyPhen_score_clean",
    "PolyPhen_boundary_distance",
    "CADD_phred_clean",
    "CADD_boundary_distance",
    "REVEL_score_clean",
    "REVEL_boundary_distance"
]].head()

In [ ]:
boundary_comparison = pd.DataFrame({
    "Predictor": ["SIFT", "PolyPhen-2", "CADD PHRED", "REVEL"],
    "Mean Distance When Correct": [
        failure_df.loc[failure_df["SIFT_failed"] == False, "SIFT_boundary_distance"].mean(),
        failure_df.loc[failure_df["PolyPhen_failed"] == False, "PolyPhen_boundary_distance"].mean(),
        failure_df.loc[failure_df["CADD_failed"] == False, "CADD_boundary_distance"].mean(),
        failure_df.loc[failure_df["REVEL_failed"] == False, "REVEL_boundary_distance"].mean()
    ],
    "Mean Distance When Incorrect": [
        failure_df.loc[failure_df["SIFT_failed"] == True, "SIFT_boundary_distance"].mean(),
        failure_df.loc[failure_df["PolyPhen_failed"] == True, "PolyPhen_boundary_distance"].mean(),
        failure_df.loc[failure_df["CADD_failed"] == True, "CADD_boundary_distance"].mean(),
        failure_df.loc[failure_df["REVEL_failed"] == True, "REVEL_boundary_distance"].mean()
    ]
})

boundary_comparison

In [ ]:
boundary_melted = boundary_comparison.melt(
    id_vars="Predictor",
    var_name="Prediction Outcome",
    value_name="Mean Boundary Distance"
)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=boundary_melted,
    x="Predictor",
    y="Mean Boundary Distance",
    hue="Prediction Outcome"
)

plt.title("Mean Distance from Decision Boundary: Correct vs Incorrect Predictions")
plt.ylabel("Mean Distance from Threshold")
plt.xlabel("Predictor")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

predictor_info = {
    "SIFT": {
        "score_col": "SIFT_score_clean",
        "failure_col": "SIFT_failed",
        "threshold": 0.05
    },
    "PolyPhen-2": {
        "score_col": "PolyPhen_score_clean",
        "failure_col": "PolyPhen_failed",
        "threshold": 0.909
    },
    "CADD PHRED": {
        "score_col": "CADD_phred_clean",
        "failure_col": "CADD_failed",
        "threshold": 20
    },
    "REVEL": {
        "score_col": "REVEL_score_clean",
        "failure_col": "REVEL_failed",
        "threshold": 0.5
    }
}

status_colors = {
    False: "steelblue",
    True: "tomato"
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (predictor, info) in zip(axes, predictor_info.items()):

    plot_df = failure_df[
        [info["score_col"], info["failure_col"]]
    ].dropna().copy()

    sns.kdeplot(
        data=plot_df,
        x=info["score_col"],
        hue=info["failure_col"],
        fill=True,
        common_norm=False,
        alpha=0.35,
        palette=status_colors,
        ax=ax,
        legend=False
    )

    ax.axvline(
        info["threshold"],
        linestyle="--",
        color="black",
        linewidth=2
    )

    ax.set_title(f"{predictor} Score Distribution", fontsize=13)
    ax.set_xlabel(f"{predictor} Score")
    ax.set_ylabel("Density")

# Shared legend below the graph
legend_elements = [
    Patch(facecolor="steelblue", alpha=0.35, label="Correct prediction"),
    Patch(facecolor="tomato", alpha=0.35, label="Incorrect / failed prediction"),
    Line2D([0], [0], color="black", linestyle="--", linewidth=2, label="Decision threshold")
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=3,
    fontsize=12,
    frameon=True,
    bbox_to_anchor=(0.5, -0.02)
)

fig.suptitle(
    "Predictor Score Distributions: Correct vs Incorrect Predictions",
    fontsize=16,
    y=0.98
)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

In [ ]:
summary_results = {
    "Total variants in test-set failure_df": len(failure_df),
    "Variants with any predictor failure": failure_df["any_failure"].sum(),
    "Any failure rate": failure_df["any_failure"].mean(),
    "Mean predictors failed per variant": failure_df["num_predictors_failed"].mean(),
    "Predictor disagreement rate": failure_df["predictor_disagreement"].mean(),
    "Failure rate among disagreement cases": failure_df.loc[
        failure_df["predictor_disagreement"],
        "any_failure"
    ].mean(),
    "Failure rate among agreement cases": failure_df.loc[
        ~failure_df["predictor_disagreement"],
        "any_failure"
    ].mean()
}

summary_df = pd.DataFrame(
    list(summary_results.items()),
    columns=["Metric", "Value"]
)

summary_df

In [ ]:
from pathlib import Path

def find_data_dir() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        candidate = base / "Data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find Data/ directory. Run the notebook from the repo root, Models/, or DataExploration/."
    )

DATA_DIR = find_data_dir()


gene_failure.to_csv(DATA_DIR / "gene_failure_analysis.csv", index=False)
top_genes.to_csv(DATA_DIR / "top_genes_phase4.csv", index=False)

substitution_failure.to_csv(DATA_DIR / "amino_acid_substitution_failure_analysis.csv", index=False)
top_substitution_failures.to_csv(DATA_DIR / "top_amino_acid_substitution_failures.csv", index=False)

biochemical_failure.to_csv(DATA_DIR / "biochemical_failure_analysis.csv", index=False)
top_biochemical_failures.to_csv(DATA_DIR / "top_biochemical_failures.csv", index=False)

disease_failure.to_csv(DATA_DIR / "disease_failure_analysis.csv", index=False)
top_disease_failures.to_csv(DATA_DIR / "top_disease_failures.csv", index=False)

disagreement_summary.to_csv(DATA_DIR / "predictor_disagreement_summary.csv", index=False)
vote_pattern_summary.to_csv(DATA_DIR / "predictor_vote_pattern_summary.csv", index=False)

boundary_comparison.to_csv(DATA_DIR / "confidence_boundary_analysis.csv", index=False)

summary_df.to_csv(DATA_DIR / "phase_4_5_6_summary.csv", index=False)

print("Saved all Phase 4–6 analysis CSV files.")


In [ ]:
for pred_col, predictor in [
    ("SIFT_pred_binary", "SIFT"),
    ("PolyPhen_pred_binary", "PolyPhen"),
    ("CADD_pred_binary", "CADD"),
    ("REVEL_pred_binary", "REVEL")
]:
    failure_df[f"{predictor}_false_positive"] = (
        (failure_df["y"] == 0) & (failure_df[pred_col] == 1)
    )

    failure_df[f"{predictor}_false_negative"] = (
        (failure_df["y"] == 1) & (failure_df[pred_col] == 0)
    )

fp_fn_summary = pd.DataFrame({
    "Predictor": ["SIFT", "PolyPhen", "CADD", "REVEL"],
    "False Positives": [
        failure_df["SIFT_false_positive"].sum(),
        failure_df["PolyPhen_false_positive"].sum(),
        failure_df["CADD_false_positive"].sum(),
        failure_df["REVEL_false_positive"].sum()
    ],
    "False Negatives": [
        failure_df["SIFT_false_negative"].sum(),
        failure_df["PolyPhen_false_negative"].sum(),
        failure_df["CADD_false_negative"].sum(),
        failure_df["REVEL_false_negative"].sum()
    ]
})

fp_fn_summary

In [ ]:
top_genes_min20 = (
    gene_failure[gene_failure["total_variants"] >= 20]
    .sort_values(
        ["any_failure_rate", "avg_predictors_failed", "total_variants"],
        ascending=False
    )
    .head(15)
)

top_genes_min20

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_genes_min20,
    x="any_failure_rate",
    y="Gene"
)

plt.title("Top Failure Genes with At Least 20 Variants")
plt.xlabel("Any Predictor Failure Rate")
plt.ylabel("Gene")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()